# Letterboxd Rating Prediction: Ridge Regression Baseline

**Objective:** predict a user's Letterboxd star rating (0.5-5.0) for a film from metadata and text features sourced from TMDB, establishing a baseline model to benchmark future iterations against.

**Approach**

1. Load and clean the raw Letterboxd ratings export (`inputs/ratings.csv`).
2. Enrich each film with TMDB metadata (budget, revenue, genres, cast, overview, keywords, etc.).
3. Derive additional numeric features (log-scaled budget/revenue/vote count, genre/country counts).
4. Split chronologically into train/test sets (most recent 20% of ratings held out) to mimic a realistic forecasting setting rather than a random, leakage-prone split.
5. Fit a `Ridge` regression pipeline over scaled numeric, one-hot categorical, and TF-IDF text features.
6. Evaluate on the held-out set with MAE / RMSE / R², rank correlations, and star-snapped accuracy.

The fitted pipeline is persisted to `models/letterboxd_tmdb_rating_model.joblib` and the enriched feature table to `outputs/ratings_with_tmdb_features.csv`, so downstream analysis doesn't need to re-run the TMDB enrichment step.

## Setup

Import the pipeline stages from `film_recommendations` and configure logging so the `INFO`-level diagnostics the package emits (row counts, evaluation metrics, etc.) are visible inline.

In [ ]:
import logging

import joblib

from film_recommendations.cleaning import clean_data
from film_recommendations.data_access import get_film_ratings
from film_recommendations.feature_extraction import create_features
from film_recommendations.ml_fitting import (
    create_transformed_features,
    evaluate_rating_predictions,
    fit_pipeline,
    make_predictions,
    split_train_test,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Load and clean ratings

Load the raw Letterboxd export and normalize it: parse `Date`, coerce `Year`/`Rating` to numeric, and drop rows with a missing name or rating.

In [ ]:
film_ratings_df = get_film_ratings()
df = clean_data(film_ratings_df)
df.head()

## 2. Feature engineering: TMDB enrichment

For each film, look up its TMDB entry and extract modeling features: runtime, budget, revenue, popularity, vote stats, genres, production countries, keywords, overview, and top cast/director. Films TMDB can't resolve are tracked in `missing` rather than silently dropped.

The enriched table is cached to `outputs/ratings_with_tmdb_features.csv` so later steps (and future notebooks) can be re-run without re-hitting the TMDB API.

> **Note:** this cell makes live TMDB API calls and can take a while to run on a cold cache.

In [ ]:
features_df, missing = create_features(df)
features_df.to_csv("../outputs/ratings_with_tmdb_features.csv", index=False)

print(f"Enriched {len(features_df)} / {len(df)} films ({len(missing)} not found on TMDB)")

## 3. Feature transformation

Derive additional numeric features from the raw TMDB fields (log-scaled budget/revenue/vote count, genre/production-country counts) and fill missing text fields, then split out the `Rating` target.

In [ ]:
features_df, target_series = create_transformed_features(features_df)
features_df.describe()

## 4. Train / test split

Split chronologically by `Date`, holding out the most recent 20% of ratings as the test set. This avoids leaking future information into training and better reflects how the model would be used in practice (predicting ratings for films watched after the training period).

In [ ]:
X_train, y_train, X_test, y_test = split_train_test(features_df)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")

## 5. Fit the Ridge regression pipeline

The pipeline combines:

- **Numeric features** (runtime, popularity, vote stats, log-scaled budget/revenue/vote count, genre/country counts) — scaled with `StandardScaler`.
- **Categorical feature** (`original_language`) — one-hot encoded.
- **Text features** (overview, keywords, cast/director, genres, production countries) — TF-IDF vectorized.

...feeding into a `Ridge` regressor. The fitted pipeline is saved to `models/letterboxd_tmdb_rating_model.joblib`.

In [ ]:
pipe = fit_pipeline(X_train, y_train, save_path="../models/letterboxd_tmdb_rating_model.joblib")
pipe

## 6. Predict on the holdout set

Generate predictions for the test set, clipped to the valid rating range \[0.5, 5.0\].

In [ ]:
eval_df, pred_clipped = make_predictions(pipe, X_test, y_test)
eval_df.head()

## 7. Evaluate

Snap predictions to the nearest half-star and compute headline accuracy metrics, plus mean absolute error broken down by true rating bucket.

In [ ]:
results = evaluate_rating_predictions(y_test, pred_clipped, eval_df)

print(f"Exact match rate (nearest 0.5 star): {results['exact_match_rate']:.1%}")
print(f"Within one snap step (0.5 star):     {results['within_half_star']:.1%}")
results["bucket_mae_by_rating"]

### Largest misses

Inspecting the biggest over- and under-predictions is useful for spotting systematic biases (e.g. genre or franchise effects the model hasn't captured).

In [ ]:
print("Largest under-predictions:")
display(eval_df.sort_values("err").head(10))

print("Largest over-predictions:")
display(eval_df.sort_values("err").tail(10))